# Modulo 8 · Lezione 2 — Neuroni Artificiali: Pesi, Bias e Attivazioni

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Anatomia di un neurone artificiale |
| 2 | Pesi e bias: cosa rappresentano |
| 3 | Funzioni di attivazione: perché ci servono |
| 4 | ReLU, Sigmoid, Tanh, Softmax in dettaglio |
| 5 | Come fluiscono i dati in una rete (forward pass) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Librerie pronte")

---
## 1. Anatomia di un neurone artificiale

```
Ogni neurone fa esattamente 2 cose:

  PASSO 1 — Combinazione lineare:
    z = w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ + b
    z = W·x + b
    (prodotto scalare tra pesi e ingressi + bias)

  PASSO 2 — Funzione di attivazione:
    a = f(z)
    (introduce la NON LINEARITÀ nella rete)

L'uscita 'a' di questo neurone diventa l'ingresso dello strato successivo.

Senza funzione di attivazione:
  z = W·x + b → una rete di 100 strati equivale a UN SOLO strato lineare.
  f(g(x)) = composizione di funzioni lineari = lineare → non guadagna complessità.

Con funzione di attivazione non lineare:
  La rete può imparare funzioni arbitrariamente complesse.
  (Teorema di approssimazione universale: una rete con abbastanza neuroni
  può approssimare QUALSIASI funzione continua)
```

In [ ]:
# Visualizzare un neurone
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('Anatomia di un neurone artificiale', fontsize=12, fontweight='bold')

# Ingressi
entradas = [(1, 5, 'x₁=2'), (1, 3.5, 'x₂=-1'), (1, 2, 'x₃=0.5')]
pesos_texto = ['w₁=0.3', 'w₂=-0.5', 'w₃=0.8']

for (x, y, txt), w_txt in zip(entradas, pesos_texto):
    circ = plt.Circle((x, y), 0.35, color='lightgreen', ec='gray', lw=1.5, zorder=5)
    ax.add_patch(circ)
    ax.text(x, y, txt, ha='center', va='center', fontsize=9)
    ax.annotate('', xy=(4.35, 3.5), xytext=(x+0.35, y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))
    mx, my = (x+0.35 + 4.35)/2 - 0.2, (y + 3.5)/2
    ax.text(mx, my, w_txt, ha='center', va='center', fontsize=8,
            color='darkorange', fontweight='bold')

# Bias
ax.text(2.5, 1, 'b = 0.1', ha='center', va='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', ec='orange'), color='darkorange')
ax.annotate('', xy=(4.35, 3.2), xytext=(3.0, 1.2),
            arrowprops=dict(arrowstyle='->', color='orange', lw=1.2, ls='dashed'))

# Somma
suma = plt.Circle((4.7, 3.5), 0.55, color='lightblue', ec='steelblue', lw=2, zorder=5)
ax.add_patch(suma)
z_val = 0.3*2 + (-0.5)*(-1) + 0.8*0.5 + 0.1
ax.text(4.7, 3.7, 'Σ', ha='center', va='center', fontsize=14)
ax.text(4.7, 3.2, f'z={z_val:.1f}', ha='center', va='center', fontsize=8)

ax.annotate('', xy=(6.45, 3.5), xytext=(5.25, 3.5),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2))
ax.text(5.85, 3.75, 'z', ha='center', fontsize=11, color='steelblue')

# Attivazione
act = plt.Circle((7.0, 3.5), 0.55, color='lightsalmon', ec='tomato', lw=2, zorder=5)
ax.add_patch(act)
a_val = max(0, z_val)  # ReLU
ax.text(7.0, 3.7, 'f(z)', ha='center', va='center', fontsize=10)
ax.text(7.0, 3.2, f'a={a_val:.1f}', ha='center', va='center', fontsize=8)
ax.text(7.0, 2.5, '(ReLU)', ha='center', fontsize=8, color='gray')

ax.annotate('', xy=(9.0, 3.5), xytext=(7.55, 3.5),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=2))
ax.text(8.75, 3.75, 'a (uscita)', ha='center', fontsize=10, color='tomato')

# Formula
ax.text(5.0, 0.7, 'z = w₁·x₁ + w₂·x₂ + w₃·x₃ + b    →    a = ReLU(z) = max(0, z)',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='white', ec='gray'))

plt.tight_layout(); plt.show()

---
## 2. Pesi e bias: cosa rappresentano

```
PESI (weights) wᵢ:
  Determinano quanto conta ciascun ingresso.
  w grande e positivo → quell'ingresso attiva il neurone
  w grande e negativo → quell'ingresso inibisce il neurone
  w ≈ 0 → quell'ingresso è irrilevante per il neurone

  In una rete di 784→128 neuroni: 784×128 = 100 352 pesi
  In GPT-3: 175 000 000 000 parametri (175 miliardi)

BIAS b:
  Permette di spostare l'attivazione indipendentemente dall'ingresso.
  Senza bias: la frontiera di decisione passa SEMPRE per l'origine.
  Con bias: la frontiera può stare ovunque.
  Analogo al termine noto in y = mx + b.

Inizializzazione:
  Se tutti i pesi sono uguali → tutti i neuroni imparano la stessa cosa (simmetria).
  Soluzione: inizializzare i pesi con piccoli valori casuali.
  Metodi: Glorot/Xavier (per Tanh), He (per ReLU).
```

In [ ]:
# Effetto del peso e del bias su un neurone semplice
x = np.linspace(-5, 5, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Effetto del peso
for w in [0.5, 1.0, 2.0, -1.0]:
    z = w * x
    a = np.maximum(0, z)  # ReLU
    axes[0].plot(x, a, label=f'w={w}, b=0', lw=2)
axes[0].set_title('Effetto del peso (b=0 fisso)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('ReLU(w·x)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# 2. Effetto del bias
for b in [-2, 0, 1, 3]:
    z = x + b  # w=1 fisso
    a = np.maximum(0, z)
    axes[1].plot(x, a, label=f'w=1, b={b}', lw=2)
axes[1].set_title('Effetto del bias (w=1 fisso)')
axes[1].set_xlabel('x'); axes[1].set_ylabel('ReLU(x + b)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# 3. Inizializzazione dei pesi
n_pesos = 10000
glorot = np.random.uniform(-np.sqrt(6/512), np.sqrt(6/512), n_pesos)  # Glorot
he     = np.random.randn(n_pesos) * np.sqrt(2/256)  # He init
normal = np.random.randn(n_pesos) * 0.01            # N(0, 0.01)

for datos, etiqueta, color in [
    (glorot, 'Glorot/Xavier', 'steelblue'),
    (he, 'He init', 'seagreen'),
    (normal, 'N(0, 0.01)', 'coral'),
]:
    axes[2].hist(datos, bins=50, alpha=0.6, color=color, label=etiqueta, density=True)
axes[2].set_title('Distribuzioni di inizializzazione dei pesi')
axes[2].set_xlabel('Valore del peso'); axes[2].set_ylabel('Densità')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 3. Funzioni di attivazione: perché ci servono

```
Senza attivazione non lineare:
  Strato 1: a₁ = W₁·x + b₁
  Strato 2: a₂ = W₂·a₁ + b₂ = W₂·(W₁·x + b₁) + b₂ = (W₂W₁)·x + c
  → resta comunque lineare! La rete profonda = un solo strato lineare.

Con attivazione non lineare f:
  Strato 1: a₁ = f(W₁·x + b₁)
  Strato 2: a₂ = f(W₂·a₁ + b₂)
  → composizione di funzioni non lineari = vera capacità espressiva.

Proprietà desiderabili:
  1. Non lineare (perché la rete sia espressiva)
  2. Differenziabile (per poter fare la backpropagation)
  3. Non satura facilmente (evita il gradiente evanescente)
  4. Computazionalmente economica
```

---
## 4. ReLU, Sigmoid, Tanh, Softmax

In [ ]:
z = np.linspace(-6, 6, 400)

# Implementare le funzioni di attivazione
def relu(z):    return np.maximum(0, z)
def leaky_relu(z, alpha=0.1): return np.where(z >= 0, z, alpha * z)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def tanh(z):    return np.tanh(z)

# Derivate
def d_relu(z):    return (z > 0).astype(float)
def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)
def d_tanh(z):    return 1 - np.tanh(z)**2

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

configs = [
    ('ReLU', relu, d_relu, 'steelblue',
     'ReLU(z) = max(0,z)\n\nPro: semplice, non satura z>0\nPro: gradiente costante\nContro: "neurone morto" z<0'),
    ('Leaky ReLU', leaky_relu, None, 'teal',
     'LReLU(z) = z se z≥0, αz se z<0\n\nPro: non muore mai\nα=0.01 tipico'),
    ('Sigmoid', sigmoid, d_sigmoid, 'tomato',
     'σ(z) = 1/(1+e^-z) ∈ (0,1)\n\nPro: uscita come probabilità\nContro: satura (gradiente ≈0)\nUso: classificazione binaria (ultimo strato)'),
    ('Tanh', tanh, d_tanh, 'seagreen',
     'tanh(z) ∈ (-1,1)\n\nPro: centrata in 0\nContro: satura come sigmoid\nUso: RNN'),
]

for col, (nombre, f, df, color, notas) in enumerate(configs):
    # Funzione
    axes[0, col].plot(z, f(z), color=color, lw=2.5)
    axes[0, col].axhline(0, color='black', lw=0.5, ls='--')
    axes[0, col].axvline(0, color='black', lw=0.5, ls='--')
    axes[0, col].set_title(nombre, fontsize=11, fontweight='bold', color=color)
    axes[0, col].set_xlabel('z'); axes[0, col].grid(True, alpha=0.3)

    # Note
    axes[1, col].text(0.05, 0.95, notas, transform=axes[1, col].transAxes,
                      fontsize=9, va='top', fontfamily='monospace',
                      bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    # Derivata
    if df is not None:
        axes[1, col].plot(z, df(z), color=color, lw=2, ls='--', alpha=0.8, label="f'(z)")
    elif nombre == 'Leaky ReLU':
        axes[1, col].plot(z, (z >= 0).astype(float) + 0.01*(z < 0), color=color, lw=2, ls='--', label="f'(z)")
    axes[1, col].axhline(0, color='black', lw=0.5)
    axes[1, col].set_xlabel('z'); axes[1, col].legend(fontsize=8)
    axes[1, col].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('f(z)')
axes[1, 0].set_ylabel('f\'(z) e note')
plt.suptitle('Funzioni di attivazione e le loro derivate', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Softmax: attivazione per il multiclasse
def softmax(z):
    # Trucco numerico: sottrarre il massimo evita l'overflow
    e = np.exp(z - np.max(z))
    return e / e.sum()

print("Softmax — converte i logit in probabilità che sommano a 1")
print("Uso: UNICO strato di uscita nella classificazione multiclasse\n")

ejemplos = [
    np.array([2.0, 1.0, 0.1]),           # classe 0 chiara
    np.array([0.5, 0.5, 0.5]),           # incertezza totale
    np.array([10.0, 2.0, 0.5]),          # classe 0 molto sicura
    np.array([1.0, 2.0, 3.0]),           # classe 2
]
clases = ['gatto', 'cane', 'uccello']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, logits in zip(axes, ejemplos):
    proba = softmax(logits)
    bars  = ax.bar(clases, proba, color=['steelblue','coral','seagreen'])
    ax.set_ylim(0, 1.05)
    ax.set_title(f'logits={np.round(logits,1)}', fontsize=9)
    ax.set_ylabel('Probabilità')
    for bar, p in zip(bars, proba):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{p:.2f}', ha='center', fontsize=9)
    ax.text(0.5, 0.92, f'Somma={proba.sum():.2f}', transform=ax.transAxes,
            ha='center', fontsize=8, color='gray')

plt.suptitle('Softmax: logit → probabilità (sommano sempre a 1)', fontsize=10)
plt.tight_layout(); plt.show()

---
## 5. Forward pass: come fluiscono i dati

```
Forward pass = propagazione in avanti
I dati fluiscono dall'ingresso all'uscita.

Per una rete con 2 strati nascosti:

  Ingresso: x  (vettore di n_features)

  Strato 1:  z₁ = W₁·x + b₁
             a₁ = ReLU(z₁)

  Strato 2:  z₂ = W₂·a₁ + b₂
             a₂ = ReLU(z₂)

  Uscita:  z_out = W_out·a₂ + b_out
           ŷ = Softmax(z_out)  (multiclasse) o Sigmoid(z_out) (binaria)

  Perdita: L = -Σ yᵢ·log(ŷᵢ)  (cross-entropy)

L'obiettivo dell'addestramento:
  Aggiustare TUTTI i pesi e i bias per minimizzare L.
  Come → Backpropagation (Lezione 3).
```

In [ ]:
# Forward pass da zero — rete 2→3→2
np.random.seed(42)

class RedNeuronal:
    """MLP semplice implementato da zero (solo forward pass per ora)"""

    def __init__(self, capas):
        """capas: lista di interi, es. [2, 4, 3] = 2 ingressi, 4 nascosti, 3 uscite"""
        self.pesos = []
        self.biases = []
        for i in range(len(capas)-1):
            # Inizializzazione He (per ReLU)
            W = np.random.randn(capas[i+1], capas[i]) * np.sqrt(2.0 / capas[i])
            b = np.zeros(capas[i+1])
            self.pesos.append(W)
            self.biases.append(b)

    def forward(self, x, verbose=False):
        a = x
        for i, (W, b) in enumerate(zip(self.pesos, self.biases)):
            z = W @ a + b
            es_ultima = (i == len(self.pesos) - 1)
            if es_ultima:
                e = np.exp(z - z.max())
                a = e / e.sum()  # Softmax
                nombre_act = 'Softmax'
            else:
                a = np.maximum(0, z)  # ReLU
                nombre_act = 'ReLU'
            if verbose:
                print(f"  Strato {i+1}: z={z.round(3)}, attivazione={nombre_act}, a={a.round(3)}")
        return a


# Demo: rete 2→4→3 con un esempio
red = RedNeuronal([2, 4, 3])
x_ejemplo = np.array([1.5, -0.5])

print("Forward pass nella rete 2→4→3:")
print(f"Ingresso: {x_ejemplo}")
salida = red.forward(x_ejemplo, verbose=True)
print(f"\nUscita (probabilità): {salida.round(4)}")
print(f"Classe predetta: {np.argmax(salida)}")
print(f"Somma delle probabilità: {salida.sum():.6f}  ← sempre 1")

In [ ]:
# Verificare le forme matriciali in una rete più grande
print("Forme matriciali nel forward pass:")
print(f"{'Strato':15} {'Ingresso':15} {'W shape':15} {'b shape':10} {'Uscita'}")
print("-" * 70)

capas = [784, 128, 64, 10]  # rete per MNIST
nombres = ['FC1 (ReLU)', 'FC2 (ReLU)', 'Uscita (Softmax)']
for i in range(len(capas)-1):
    ent = capas[i]; sal = capas[i+1]
    n_params = ent * sal + sal
    print(f"{nombres[i]:15} ({ent},):          ({sal},{ent}):     ({sal},):    ({sal},)  → {n_params:,} params")

total = sum(capas[i]*capas[i+1] + capas[i+1] for i in range(len(capas)-1))
print(f"\nTotale parametri: {total:,}")
print(f"Memoria (float32): {total*4/1024:.1f} KB")

---
## Pratica

### Esercizio 1 — Implementare funzioni di attivazione

**Consegna:** Implementa le seguenti funzioni e le loro derivate:

1. `elu(z, alpha=1.0)` → ELU: `z se z>0`, `alpha*(exp(z)-1) se z≤0`
2. `swish(z)` → Swish: `z * sigmoid(z)`
3. `gelu(z)` → GELU (approssimazione): `0.5 * z * (1 + tanh(sqrt(2/π) * (z + 0.044715 * z³)))`

Plottale insieme a ReLU e commenta le loro differenze.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-4, 4, 400)

def relu(z):   return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def elu(z, alpha=1.0):  return np.where(z > 0, z, alpha * (np.exp(z) - 1))
def swish(z):  return z * sigmoid(z)
def gelu(z):   return 0.5 * z * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3)))

fig, ax = plt.subplots(figsize=(9, 4))
for f, nombre, color in [
    (relu, 'ReLU', 'steelblue'),
    (elu,  'ELU', 'tomato'),
    (swish,'Swish', 'seagreen'),
    (gelu, 'GELU', 'mediumpurple'),
]:
    ax.plot(z, f(z), label=nombre, lw=2)

ax.axhline(0, color='black', lw=0.5, ls='--')
ax.axvline(0, color='black', lw=0.5, ls='--')
ax.set_xlabel('z'); ax.set_ylabel('f(z)')
ax.set_title('Confronto tra funzioni di attivazione moderne')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(-1.5, 4)
plt.tight_layout(); plt.show()

print("ReLU:  semplice, veloce, può 'uccidere' neuroni (z<0 → gradiente=0)")
print("ELU:   liscia in z<0, permette gradienti non nulli → non muore")
print("Swish: liscia, non monotona, funziona meglio in reti molto profonde (Google 2017)")
print("GELU:  usata in BERT e GPT. Liscia e stocastica nella sua versione originale")
```
</details>

---
### Esercizio 2 — Problema del gradiente evanescente

**Consegna:** Dimostra perché Sigmoid causa il problema del gradiente evanescente nelle reti profonde:

1. La derivata di Sigmoid è: `σ'(z) = σ(z)·(1-σ(z))`
2. Il valore massimo di σ' è 0.25 (in z=0)
3. Simula la backprop: moltiplica 20 derivate di sigmoid con ingresso z=0
4. Fai lo stesso con ReLU (derivata = 1 se z>0)
5. Perché ReLU risolve il problema?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z): return 1 / (1 + np.exp(-z))
def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)
def d_relu(z): return float(z > 0)

n_capas = 30
z_val   = 0.5  # ingresso a ogni strato

grad_sigmoid = [1.0]
grad_relu    = [1.0]

for _ in range(n_capas):
    grad_sigmoid.append(grad_sigmoid[-1] * d_sigmoid(z_val))
    grad_relu.append(grad_relu[-1] * d_relu(z_val))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(range(n_capas+1), grad_sigmoid, 'r-o', ms=4, label='Sigmoid')
axes[0].semilogy(range(n_capas+1), grad_relu,    'b-o', ms=4, label='ReLU')
axes[0].set_xlabel('Numero di strati'); axes[0].set_ylabel('Magnitudine del gradiente (log)')
axes[0].set_title('Gradiente evanescente: Sigmoid vs ReLU')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].text(0.05, 0.95,
    f"Gradiente Sigmoid dopo {n_capas} strati:\n  {grad_sigmoid[-1]:.2e}\n\n"
    f"Gradiente ReLU dopo {n_capas} strati:\n  {grad_relu[-1]:.2f}\n\n"
    f"d_sigmoid(0.5) = {d_sigmoid(z_val):.4f} < 0.25\n"
    f"d_relu(0.5)    = {d_relu(z_val):.4f}\n\n"
    f"Dopo {n_capas} strati:\n"
    f"  Sigmoid: 0.25^{n_capas} = {0.25**n_capas:.2e}\n"
    f"  → i pesi dei primi strati\n"
    f"    non imparano nulla (gradiente ≈ 0)\n\n"
    f"  ReLU: 1^{n_capas} = 1.0\n"
    f"  → il gradiente fluisce senza attenuarsi",
    transform=axes[1].transAxes, fontsize=10, va='top',
    fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[1].axis('off')
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 3 — Forward pass manuale

**Consegna:** Dato questo MLP con pesi noti:

```
Architettura: 3 → 2 → 1 (classificazione binaria)

W1 = [[0.5, -0.2, 0.1],    b1 = [0.0, 0.5]
      [0.3,  0.8, -0.5]]

W2 = [[0.7, -0.3]]         b2 = [0.1]

Ingresso: x = [1.0, 2.0, -1.0]
Strati nascosti: ReLU. Uscita: Sigmoid.
```

1. Calcola a mano ogni passo (z₁, a₁, z₂, a₂)
2. Verifica con il codice
3. Qual è la probabilità predetta della classe 1?
4. Quale classe predice con soglia 0.5?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np

# Pesi
W1 = np.array([[0.5, -0.2, 0.1],
               [0.3,  0.8, -0.5]])
b1 = np.array([0.0, 0.5])
W2 = np.array([[0.7, -0.3]])
b2 = np.array([0.1])

x = np.array([1.0, 2.0, -1.0])

# Forward pass passo dopo passo
z1 = W1 @ x + b1
print(f"z₁ = W1·x + b1 = {z1.round(4)}")

a1 = np.maximum(0, z1)  # ReLU
print(f"a₁ = ReLU(z₁) = {a1.round(4)}")

z2 = W2 @ a1 + b2
print(f"z₂ = W2·a₁ + b2 = {z2.round(4)}")

a2 = 1 / (1 + np.exp(-z2))  # Sigmoid
print(f"a₂ = σ(z₂) = {a2.round(4)}")

clase = int(a2[0] >= 0.5)
print(f"\nProbabilità classe 1: {a2[0]:.4f}")
print(f"Classe predetta (soglia 0.5): {clase}")

# Verifica manuale:
# z1[0] = 0.5*1 - 0.2*2 + 0.1*(-1) + 0 = 0.5 - 0.4 - 0.1 = 0.0
# z1[1] = 0.3*1 + 0.8*2 - 0.5*(-1) + 0.5 = 0.3 + 1.6 + 0.5 + 0.5 = 2.9
print(f"\nVerifica manuale:")
print(f"  z1[0] = 0.5·1 - 0.2·2 + 0.1·(-1) + 0 = {0.5*1 - 0.2*2 + 0.1*(-1) + 0}")
print(f"  z1[1] = 0.3·1 + 0.8·2 - 0.5·(-1) + 0.5 = {0.3*1 + 0.8*2 - 0.5*(-1) + 0.5}")
```
</details>

---
### Esercizio 4 — Quanti parametri ha la mia rete

**Consegna:** Implementa `contar_parametros(capas)` che restituisce il numero totale di parametri (pesi + bias) di un MLP completamente connesso.

Calcola e confronta:
- Rete A: [784, 128, 10]
- Rete B: [784, 512, 256, 128, 10]
- Rete C: [784, 1024, 1024, 1024, 10]
- AlexNet approssimata: [150528, 4096, 4096, 1000]

Quanta RAM occupa ciascuna in float32 (4 byte per parametro)?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
def contar_parametros(capas):
    total = 0
    detalle = []
    for i in range(len(capas) - 1):
        pesos = capas[i] * capas[i+1]
        biases = capas[i+1]
        total += pesos + biases
        detalle.append((f'Strato {i+1} ({capas[i]}→{capas[i+1]})', pesos, biases))
    return total, detalle

redes = [
    ('Rete A (piccola)',   [784, 128, 10]),
    ('Rete B (media)',     [784, 512, 256, 128, 10]),
    ('Rete C (grande)',    [784, 1024, 1024, 1024, 10]),
    ('AlexNet FC (approx)',[150528, 4096, 4096, 1000]),
]

print(f"{'Rete':25} {'Parametri':>15} {'RAM (MB)':>10}")
print('-' * 55)
for nombre, capas in redes:
    total, _ = contar_parametros(capas)
    mb = total * 4 / 1024 / 1024
    print(f"{nombre:25} {total:>15,} {mb:>10.2f}")

print("\nDettaglio Rete B:")
_, detalle = contar_parametros([784, 512, 256, 128, 10])
for capa, pesos, biases in detalle:
    print(f"  {capa}: {pesos:,} pesi + {biases} bias = {pesos+biases:,}")
```
</details>

---
### Esercizio 5 (sfida) — MLP completo con forward pass a batch

**Consegna:** Estendi la classe `RedNeuronal` definita sopra affinché:

1. Accetti un batch (matrice di campioni) invece di un solo vettore
2. Calcoli il forward pass per tutti i campioni contemporaneamente (operazioni matriciali)
3. Implementi `cross_entropy_loss(y_pred, y_true)` per il multiclasse
4. Provala con un batch di 32 campioni da 4 feature, 3 classi
5. Verifica che la somma delle probabilità di ogni campione sia = 1

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np

class MLPBatch:
    def __init__(self, capas):
        self.pesos, self.biases = [], []
        for i in range(len(capas)-1):
            W = np.random.randn(capas[i], capas[i+1]) * np.sqrt(2.0/capas[i])
            b = np.zeros(capas[i+1])
            self.pesos.append(W)
            self.biases.append(b)

    def forward(self, X):
        # X: (batch_size, n_features)
        a = X
        for i, (W, b) in enumerate(zip(self.pesos, self.biases)):
            z = a @ W + b                 # (batch, out)
            es_ultima = (i == len(self.pesos)-1)
            if es_ultima:
                z -= z.max(axis=1, keepdims=True)  # stabilità numerica
                e = np.exp(z)
                a = e / e.sum(axis=1, keepdims=True)  # softmax per riga
            else:
                a = np.maximum(0, z)  # ReLU
        return a  # (batch, n_clases)

    @staticmethod
    def cross_entropy_loss(y_pred, y_true):
        # y_true: one-hot (batch, n_clases)
        # y_pred: probabilità (batch, n_clases)
        eps = 1e-9
        return -np.mean(np.sum(y_true * np.log(y_pred + eps), axis=1))


np.random.seed(42)
batch_size = 32
n_features = 4
n_clases   = 3

red_batch = MLPBatch([n_features, 8, 8, n_clases])
X_batch   = np.random.randn(batch_size, n_features)
y_int     = np.random.randint(0, n_clases, batch_size)
y_onehot  = np.eye(n_clases)[y_int]  # one-hot

y_pred = red_batch.forward(X_batch)

print(f"X_batch shape: {X_batch.shape}")
print(f"y_pred shape:  {y_pred.shape}")
print(f"\nPrime 3 probabilità:")
print(y_pred[:3].round(4))
print(f"\nSomma per campione (devono essere ≈1): {y_pred.sum(axis=1)[:5].round(6)}")

loss = MLPBatch.cross_entropy_loss(y_pred, y_onehot)
print(f"\nCross-entropy loss: {loss:.4f}")
print(f"Loss di un modello casuale (3 classi): log(3)={np.log(3):.4f}")
```
</details>

---
## Riepilogo

```python
import numpy as np

# Un neurone: combinazione lineare + attivazione
z = np.dot(W, x) + b          # somma pesata
a = np.maximum(0, z)          # ReLU

# Funzioni di attivazione
relu    = lambda z: np.maximum(0, z)         # strati nascosti → default
sigmoid = lambda z: 1/(1+np.exp(-z))         # uscita binaria (0,1)
softmax = lambda z: np.exp(z)/np.exp(z).sum()  # uscita multiclasse
tanh    = lambda z: np.tanh(z)               # RNN

# Forward pass (batch)
# X: (batch, features), W: (features, neuroni), b: (neuroni,)
z = X @ W + b                 # (batch, neuroni)
a = np.maximum(0, z)          # ReLU su tutto il batch

# Regola d'oro delle attivazioni:
# Strati nascosti → ReLU (o LeakyReLU, GELU nei transformer)
# Uscita binaria → Sigmoid
# Uscita multiclasse → Softmax
# Uscita regressione → nessuna (lineare)
```

## Cosa viene nella prossima lezione?
Nella **Lezione 3** impareremo la **Backpropagation**: l'algoritmo che aggiusta tutti i pesi della rete usando il gradiente della perdita. È il motore dell'apprendimento nel Deep Learning.